# Paneles solares: Faster R-CNN por fuente

Dos arquitecturas de torchvision (`fasterrcnn_mobilenet_v3_large_fpn` y `fasterrcnn_resnet50_fpn`), tres estrategias de congelamiento y los mismos tres folds que YOLO. Son 18 entrenamientos. La métrica principal es el promedio macro de mAP50-95, con el mismo peso para cada fold.

Elegí una GPU NVIDIA en *Entorno de ejecución*. En Colab, el repositorio se clona desde GitHub en el disco de la sesión y los resultados (`best.pt`, CSV, gráficos) se guardan en Google Drive, que sobrevive si la sesión se corta.

Las celdas de rutas y verificación no cargan modelos. La de entrenamiento sí.

In [2]:
import os

os.environ["PYTHONHASHSEED"] = "42"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Fuera de Colab: no se montó Drive.")

Mounted at /content/drive


In [3]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ana-castillo-zuain/procesamiento-de-imagenes.git"

if IN_COLAB:
    # El repo se clona en /content: son unos segundos y no hace falta subir nada a Drive.
    # /content se borra al cortarse la sesión; por eso los resultados van a Drive.
    REPO_DIR = Path("/content/procesamiento-de-imagenes")
    if REPO_DIR.is_dir():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    RESULTS_DIR = Path("/content/drive/MyDrive/paneles_resultados/RCNN")
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "RCNN" else Path.cwd()
    RESULTS_DIR = REPO_DIR / "RCNN" / "results"

CODE_DIR = REPO_DIR / "RCNN"
DATASET_DIR = REPO_DIR / "data" / "yolo"

assert CODE_DIR.joinpath("train.py").is_file(), f"No está train.py en {CODE_DIR}"
assert REPO_DIR.joinpath("YOLO", "solar_yolo").is_dir(), "Falta YOLO/solar_yolo: solar_rcnn usa sus folds y CSV."
assert DATASET_DIR.joinpath("PV01_val").is_dir(), f"No está el dataset en {DATASET_DIR}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("código    :", CODE_DIR)
print("dataset   :", DATASET_DIR)
print("resultados:", RESULTS_DIR)

código    : /content/procesamiento-de-imagenes/RCNN
dataset   : /content/procesamiento-de-imagenes/data/yolo
resultados: /content/drive/MyDrive/paneles_resultados/RCNN


In [3]:
import torch
import torchvision

# No se instala nada: Colab ya trae torch y torchvision con CUDA.
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA no está disponible. Activá una GPU en el entorno de ejecución antes de entrenar. "
        "No se cargó ningún modelo."
    )
print(torch.cuda.get_device_name(0))
print("torch", torch.__version__, "torchvision", torchvision.__version__, "cuda", torch.version.cuda)

Tesla T4
torch 2.11.0+cu130 torchvision 0.26.0+cu130 cuda 13.0


In [4]:
import sys
sys.path.insert(0, str(CODE_DIR))

from solar_rcnn.config import Config
from solar_rcnn.engine import verify_setup

config = Config(dataset_dir=DATASET_DIR, results_dir=RESULTS_DIR, batch=8)
print(verify_setup(config))

Arquitecturas y congelamiento

fasterrcnn_mobilenet_v3_large_fpn  (pesos FasterRCNN_MobileNet_V3_Large_FPN_Weights.COCO_V1, cabeza cambiada a 2 clases)
  full     trainable_backbone_layers=6  congela: (ninguno)
  backbone trainable_backbone_layers=0  congela: 0 stem conv stride 2 (P1/2); 1 bneck stride 2; 2 bneck, baja a stride 4 (P2/4); 3 bneck stride 4; 4 bneck, baja a stride 8 (P3/8); 5 bneck stride 8; 6 bneck stride 8; 7 bneck, baja a stride 16 (P4/16); 8 bneck stride 16; 9 bneck stride 16; 10 bneck stride 16; 11 bneck stride 16; 12 bneck stride 16; 13 bneck, baja a stride 32 (P5/32), primera salida al FPN; 14 bneck stride 32; 15 bneck stride 32; 16 conv 1x1 a 960 canales, stride 32, segunda salida al FPN
  partial  trainable_backbone_layers=4  congela: 0 stem conv stride 2 (P1/2); 1 bneck stride 2; 2 bneck, baja a stride 4 (P2/4); 3 bneck stride 4
  nota: El FPN de esta red toma solo las salidas de features[13] y features[16], las dos de stride 32, más un max pool (stride 64). Con

## Entrenar

`None` recorre las 18 combinaciones. Para una sola, pasá una lista. Conviene empezar por la MobileNet, que es mucho más rápida, para ver que todo anda.

Si Colab se queda sin memoria, bajá `batch` en la celda anterior y volvé a correr desde ahí. Un experimento ya terminado, con su `best.pt`, no se repite. Uno cortado a la mitad sigue desde su última época.

In [5]:
from solar_rcnn.engine import run_experiments

SELECTED_ARCHITECTURES = None  # ["fasterrcnn_mobilenet_v3_large_fpn", "fasterrcnn_resnet50_fpn"]
SELECTED_STRATEGIES = None     # ["full", "backbone", "partial"]
SELECTED_FOLDS = None          # [1, 2, 3]

outcome = run_experiments(
    config,
    architectures=SELECTED_ARCHITECTURES,
    strategies=SELECTED_STRATEGIES,
    folds=SELECTED_FOLDS,
)
print("fallos:", len(outcome["failures"]))
for failure in outcome["failures"]:
    print(failure["experiment_id"], failure["error"])

2026-11-03 02:30:44,209 INFO 
Arquitecturas y congelamiento

fasterrcnn_mobilenet_v3_large_fpn  (pesos FasterRCNN_MobileNet_V3_Large_FPN_Weights.COCO_V1, cabeza cambiada a 2 clases)
  full     trainable_backbone_layers=6  congela: (ninguno)
  backbone trainable_backbone_layers=0  congela: 0 stem conv stride 2 (P1/2); 1 bneck stride 2; 2 bneck, baja a stride 4 (P2/4); 3 bneck stride 4; 4 bneck, baja a stride 8 (P3/8); 5 bneck stride 8; 6 bneck stride 8; 7 bneck, baja a stride 16 (P4/16); 8 bneck stride 16; 9 bneck stride 16; 10 bneck stride 16; 11 bneck stride 16; 12 bneck stride 16; 13 bneck, baja a stride 32 (P5/32), primera salida al FPN; 14 bneck stride 32; 15 bneck stride 32; 16 conv 1x1 a 960 canales, stride 32, segunda salida al FPN
  partial  trainable_backbone_layers=4  congela: 0 stem conv stride 2 (P1/2); 1 bneck stride 2; 2 bneck, baja a stride 4 (P2/4); 3 bneck stride 4
  nota: El FPN de esta red toma solo las salidas de features[13] y features[16], las dos de stride 32, má

Downloading: "https://download.pytorch.org/models/fasterrcnn_mobilenet_v3_large_fpn-fb6a3cc7.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_mobilenet_v3_large_fpn-fb6a3cc7.pth


100%|██████████| 74.2M/74.2M [00:00<00:00, 185MB/s]
2026-11-03 02:30:45,412 INFO Congelamiento verificado: 18930229 parámetros entrenables, 0 congelados.
2026-11-03 02:31:06,450 INFO fasterrcnn_mobilenet_v3_large_fpn__full__fold1 época 1/50: loss 0.5676  mAP50 0.0777  mAP50-95 0.0241  (21 s)
2026-11-03 02:31:22,091 INFO fasterrcnn_mobilenet_v3_large_fpn__full__fold1 época 2/50: loss 0.6900  mAP50 0.1859  mAP50-95 0.0602  (15 s)
2026-11-03 02:31:42,337 INFO fasterrcnn_mobilenet_v3_large_fpn__full__fold1 época 3/50: loss 0.9137  mAP50 0.1910  mAP50-95 0.0617  (19 s)
2026-11-03 02:31:59,662 INFO fasterrcnn_mobilenet_v3_large_fpn__full__fold1 época 4/50: loss 0.9042  mAP50 0.1893  mAP50-95 0.0611  (17 s)
2026-11-03 02:32:19,023 INFO fasterrcnn_mobilenet_v3_large_fpn__full__fold1 época 5/50: loss 0.8808  mAP50 0.2416  mAP50-95 0.0925  (19 s)
2026-11-03 02:32:37,755 INFO fasterrcnn_mobilenet_v3_large_fpn__full__fold1 época 6/50: loss 0.9213  mAP50 0.2787  mAP50-95 0.0924  (18 s)
2026-11-03 0

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:01<00:00, 161MB/s]
2026-11-03 03:28:09,443 INFO Congelamiento verificado: 41299161 parámetros entrenables, 0 congelados.
/content/procesamiento-de-imagenes/RCNN/solar_rcnn/engine.py:641: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
2026-11-03 03:28:33,305 INFO fasterrcnn_resnet50_fpn__full__fold1 época 1/50: loss 0.4713  mAP50 0.3316  mAP50-95 0.1404  (24 s)
2026-11-03 03:29:01,219 INFO fasterrcnn_resnet50_fpn__full__fold1 época 2/50: loss 0.2671  mAP50 0.5806  mAP50-95 0.2874  (23 s)
2026-11-03 03:29:24,225 INFO fasterrcnn_resnet50_fpn__full__fold1 época 3/50: loss 0.2146  mAP50 0.6445  mAP50-95 0

fallos: 0


In [4]:
import pandas as pd

pd.read_csv(RESULTS_DIR / "summary.csv")[
    ["architecture", "strategy", "fold1_map50_95", "fold2_map50_95", "fold3_map50_95", "macro_map50_95", "score_status"]
]

,architecture,strategy,fold1_map50_95,fold2_map50_95,fold3_map50_95,macro_map50_95,score_status
0,fasterrcnn_mobilenet_v3_large_fpn,backbone,0.030593,0.170222,0.117085,0.105967,complete
1,fasterrcnn_mobilenet_v3_large_fpn,full,0.123543,0.265851,0.122650,0.170681,complete
2,fasterrcnn_mobilenet_v3_large_fpn,partial,0.105680,0.245471,0.149346,0.166832,complete
3,fasterrcnn_resnet50_fpn,backbone,0.344063,0.328409,0.163453,0.278642,complete
4,fasterrcnn_resnet50_fpn,full,0.465808,0.323192,0.186521,0.325174,complete
5,fasterrcnn_resnet50_fpn,partial,0.436301,0.351281,0.204341,0.330641,complete
